In [3]:
import pandas as pd

# =========================
# 📥 CARREGAR ARQUIVOS
# =========================

path1 = '../Base de Dados/Bases do Kaggle/animelists_cleaned.csv'
path2 = '../Base de Dados/Bases do Kaggle/users_cleaned.csv'

animelists_cleaned = pd.read_csv(path1)
users_cleaned = pd.read_csv(path2)

In [4]:
# =========================
# ADICIONAR GÊNERO AO ANIMELISTS_CLEANED
# =========================
animelists_cleaned = animelists_cleaned.merge(
    users_cleaned[['username', 'gender']],
    on='username',
    how='left'
)

# =========================
# DROPAR COLUNAS DESNECESSÁRIAS
# =========================
animelists_cleaned = animelists_cleaned.drop(columns=[
    'my_start_date',
    'my_finish_date',
    'my_last_updated',
    'my_tags'
])

# =========================
# VERIFICAR RESULTADO
# =========================
print(animelists_cleaned.shape)
print(animelists_cleaned.head())

(31284030, 8)
   username  anime_id  my_watched_episodes  my_score  my_status  \
0  karthiga        21                  586         9          1   
1  karthiga        59                   26         7          2   
2  karthiga        74                   26         7          2   
3  karthiga       120                   26         7          2   
4  karthiga       178                   26         7          2   

   my_rewatching  my_rewatching_ep  gender  
0            NaN                 0  Female  
1            NaN                 0  Female  
2            NaN                 0  Female  
3            NaN                 0  Female  
4            0.0                 0  Female  


In [14]:
import requests
import pandas as pd
import time

# =====================================================
# CONFIGURAÇÃO
# =====================================================

CLIENT_ID = "7f703a2204ee6910eecc67347c7224e9"

HEADERS = {
    "X-MAL-CLIENT-ID": CLIENT_ID
}

BASE_URL = "https://api.myanimelist.net/v2/anime/ranking"

# Máximo permitido
LIMIT = 100

# =====================================================
# TIPOS DE RANKING
# =====================================================

# Isso aumenta MUITO a quantidade de animes
ranking_types = [
    "all",
    "airing",
    "upcoming",
    "tv",
    "ova",
    "movie",
    "special",
    "bypopularity",
    "favorite"
]

# =====================================================
# QUANTIDADE DE PÁGINAS
# =====================================================

TOTAL_PAGES = 250

# =====================================================
# LISTA FINAL
# =====================================================

anime_data = []

# Cache para evitar duplicados
anime_ids = set()

# =====================================================
# EXTRAIR DEMOGRAFIA
# =====================================================

def extract_demographic(genres_list):

    demographics = []

    for genre in genres_list:

        genre_name = genre.get('name', '')

        if genre_name in [
            'Shounen',
            'Shoujo',
            'Seinen',
            'Josei',
            'Kids'
        ]:
            demographics.append(genre_name)

    if demographics:
        return ', '.join(demographics)

    return 'Unknown'

# =====================================================
# LOOP PRINCIPAL
# =====================================================

for ranking_type in ranking_types:

    print("\n====================================")
    print(f"RANKING: {ranking_type}")
    print("====================================")

    for page in range(TOTAL_PAGES):

        offset = page * LIMIT

        print(
            f"\nPágina {page + 1}/{TOTAL_PAGES}"
        )

        print(f"Offset: {offset}")

        params = {
            'ranking_type': ranking_type,
            'limit': LIMIT,
            'offset': offset,
            'fields': (
                'id,title,mean,rank,popularity,'
                'num_list_users,num_scoring_users,'
                'media_type,status,genres,'
                'num_episodes,source,'
                'average_episode_duration'
            )
        }

        try:

            response = requests.get(
                BASE_URL,
                headers=HEADERS,
                params=params,
                timeout=30
            )

            if response.status_code != 200:

                print(
                    f"Erro {response.status_code}"
                )

                print(response.text)

                break

            data = response.json()

            if 'data' not in data:

                print('Nenhum dado.')

                break

            animes = data['data']

            if len(animes) == 0:

                print("Fim das páginas.")

                break

            print(
                f"Encontrados: {len(animes)}"
            )

            novos = 0

            for item in animes:

                node = item.get('node', {})

                anime_id = node.get('id')

                # Evita duplicados
                if anime_id in anime_ids:
                    continue

                anime_ids.add(anime_id)

                genres = node.get('genres', [])

                genre_names = [
                    g['name']
                    for g in genres
                ]

                demographic = extract_demographic(
                    genres
                )

                anime_info = {
                    'anime_id': anime_id,
                    'title': node.get('title'),
                    'type': node.get('media_type'),
                    'source': node.get('source'),
                    'episodes': node.get('num_episodes'),
                    'status': node.get('status'),
                    'duration': node.get(
                        'average_episode_duration'
                    ),
                    'score': node.get('mean'),
                    'scored_by': node.get(
                        'num_scoring_users'
                    ),
                    'rank': node.get('rank'),
                    'popularity': node.get(
                        'popularity'
                    ),
                    'members': node.get(
                        'num_list_users'
                    ),
                    'favorites': None,
                    'genre': ', '.join(
                        genre_names
                    ),
                    'demographic': demographic
                }

                anime_data.append(anime_info)

                novos += 1

            print(
                f"Novos animes: {novos}"
            )

            print(
                f"Total acumulado: "
                f"{len(anime_data)}"
            )

            # Rate limit
            time.sleep(1)

        except Exception as e:

            print(f"Erro: {e}")

            time.sleep(5)

# =====================================================
# DATAFRAME
# =====================================================

print("\nCriando DataFrame...")

anime_df = pd.DataFrame(anime_data)

anime_df = anime_df.drop_duplicates(
    subset=['anime_id']
)

# =====================================================
# ESTATÍSTICAS
# =====================================================

print('\n====================================')
print('ESTATÍSTICAS')
print('====================================')

print(
    f"\nTotal de animes: "
    f"{len(anime_df)}"
)

print('\nDemografias:\n')

print(
    anime_df['demographic']
    .value_counts(dropna=False)
)

print('\nTipos:\n')

print(
    anime_df['type']
    .value_counts(dropna=False)
)

print('\nStatus:\n')

print(
    anime_df['status']
    .value_counts(dropna=False)
)

# =====================================================
# SALVAR CSV
# =====================================================

OUTPUT_FILE = (
    '../Base de Dados/Bases Geradas/myanimelist.csv'
)

anime_df.to_csv(
    OUTPUT_FILE,
    index=False,
    encoding='utf-8-sig'
)

print(
    f"\nDataset salvo em: "
    f"{OUTPUT_FILE}"
)

# =====================================================
# VISUALIZAÇÃO
# =====================================================

print('\nPrimeiras linhas:\n')

print(anime_df.head())


RANKING: all

Página 1/250
Offset: 0
Encontrados: 100
Novos animes: 100
Total acumulado: 100

Página 2/250
Offset: 100
Encontrados: 100
Novos animes: 100
Total acumulado: 200

Página 3/250
Offset: 200
Encontrados: 100
Novos animes: 100
Total acumulado: 300

Página 4/250
Offset: 300
Encontrados: 100
Novos animes: 100
Total acumulado: 400

Página 5/250
Offset: 400
Encontrados: 100
Novos animes: 100
Total acumulado: 500

Página 6/250
Offset: 500
Encontrados: 100
Novos animes: 100
Total acumulado: 600

Página 7/250
Offset: 600
Encontrados: 100
Novos animes: 100
Total acumulado: 700

Página 8/250
Offset: 700
Encontrados: 100
Novos animes: 100
Total acumulado: 800

Página 9/250
Offset: 800
Encontrados: 100
Novos animes: 100
Total acumulado: 900

Página 10/250
Offset: 900
Encontrados: 100
Novos animes: 100
Total acumulado: 1000

Página 11/250
Offset: 1000
Encontrados: 100
Novos animes: 100
Total acumulado: 1100

Página 12/250
Offset: 1100
Encontrados: 100
Novos animes: 100
Total acumulado: 1

In [5]:
path3 = '../Base de Dados/Bases Geradas/myanimelist.csv'

MyAnimeList = pd.read_csv(path3)

In [7]:
# =========================
# FAZER O MERGE ENTRE A MYANIME LIST E A ANIMELISTS_CLEANED
# =========================
usersanimelist = animelists_cleaned.merge(
    MyAnimeList,
    on='anime_id',
    how='inner'  # pode trocar pra 'left' se quiser manter tudo da lista
)

print(usersanimelist.head())

# =========================
# VERIFICAR RESULTADO
# =========================
print(usersanimelist.shape)
print(usersanimelist.head())

   username  anime_id  my_watched_episodes  my_score  my_status  \
0  karthiga        21                  586         9          1   
1  karthiga        59                   26         7          2   
2  karthiga        74                   26         7          2   
3  karthiga       120                   26         7          2   
4  karthiga       178                   26         7          2   

   my_rewatching  my_rewatching_ep  gender          title type  ...  \
0            NaN                 0  Female      One Piece   tv  ...   
1            NaN                 0  Female        Chobits   tv  ...   
2            NaN                 0  Female   Gakuen Alice   tv  ...   
3            NaN                 0  Female  Fruits Basket   tv  ...   
4            0.0                 0  Female   Ultra Maniac   tv  ...   

             status  duration score  scored_by    rank  popularity  members  \
0  currently_airing      1440  8.73    1529415    55.0          17  2678508   
1   finished

In [13]:
# =========================
# DROPAR COLUNAS DESNECESSÁRIAS
# =========================
columns_to_drop = ['status', 'duration', 'source', 'scored_by', 'rank', 'popularity', 'members', 'favorites']
existing_columns = [col for col in columns_to_drop if col in usersanimelist.columns]
if existing_columns:
    usersanimelist = usersanimelist.drop(columns=existing_columns)

# =========================
# VERIFICAR RESULTADO
# =========================
print(usersanimelist.shape)
print(usersanimelist.head())


# =========================
# SALVAR FINAL
# =========================
usersanimelist.to_csv("../Base de Dados/Bases Geradas/usersanimelist.csv", index=False)

print("Finalizado com sucesso!")

(31264431, 14)
   username  anime_id  my_watched_episodes  my_score  my_status  \
0  karthiga        21                  586         9          1   
1  karthiga        59                   26         7          2   
2  karthiga        74                   26         7          2   
3  karthiga       120                   26         7          2   
4  karthiga       178                   26         7          2   

   my_rewatching  my_rewatching_ep  gender          title type  episodes  \
0            NaN                 0  Female      One Piece   tv         0   
1            NaN                 0  Female        Chobits   tv        26   
2            NaN                 0  Female   Gakuen Alice   tv        26   
3            NaN                 0  Female  Fruits Basket   tv        26   
4            0.0                 0  Female   Ultra Maniac   tv        26   

   score                                              genre demographic  
0   8.73                Action, Adventure, Fantasy,